# Vesuvius V11 - Inference for 2x T4 GPUs (Kaggle)

**Matching V11 Training Exactly:**
- Patch: 192³ (same as training)
- Normalization: Percentile clipping (0.5-99.5%) + Z-score
- Multi-GPU: DataParallel across 2x T4
- Precision: Float16
- TTA: flip (4 predictions)

**Post-processing: SIMPLE (threshold 0.50 + tiny component removal)**

**Score history:**
| Config | Score |
|--------|-------|
| Normalization mismatch + 128³ patches | 0.492 |
| Fixed normalization + 192³ patches | 0.531 |
| + deep supervision heads fix | 0.538 |
| + skeleton-guided threshold 0.70 | 0.471 (worse!) |
| Reverted to threshold 0.50 + simple | 0.538 |

In [ ]:
!pip install imagecodecs tifffile connected-components-3d -q

In [ ]:
import os
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

import gc
import warnings
from pathlib import Path
from typing import List, Tuple
from dataclasses import dataclass, field

import numpy as np
import pandas as pd
import tifffile
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F

from scipy.ndimage import (
    binary_fill_holes, distance_transform_edt, gaussian_filter,
    label, generate_binary_structure, binary_dilation
)

warnings.filterwarnings('ignore')

# =============================================================================
# MULTI-GPU DETECTION
# =============================================================================
N_GPUS = torch.cuda.device_count()
print(f"Available GPUs: {N_GPUS}")
for i in range(N_GPUS):
    props = torch.cuda.get_device_properties(i)
    print(f"  GPU {i}: {props.name} ({props.total_memory / 1e9:.1f} GB)")

@dataclass
class Config:
    # Paths
    TEST_ROOT: Path = Path("/kaggle/input/vesuvius-challenge-2025/test")
    CHECKPOINT_PATH: Path = Path("/kaggle/input/v11-checkpoint/fold_0/best_model.pth")
    OUTPUT_DIR: Path = Path("/kaggle/working")
    
    # Model (must match training)
    TRAIN_PATCH_SIZE: Tuple[int, int, int] = (192, 192, 192)
    FEATURES: List[int] = field(default_factory=lambda: [32, 64, 128, 256, 320, 320])
    N_BLOCKS: List[int] = field(default_factory=lambda: [1, 2, 3, 4, 6, 6])
    
    # Inference - MATCH TRAINING!
    INFER_PATCH_SIZE: Tuple[int, int, int] = (192, 192, 192)
    OVERLAP: float = 0.5
    TTA_LEVEL: str = "flip"
    USE_FLOAT16: bool = True
    
    # Multi-GPU settings
    USE_MULTI_GPU: bool = True
    BATCH_SIZE: int = 1 * N_GPUS
    
    # Post-processing - SIMPLE (best so far = 0.538)
    THRESHOLD: float = 0.50
    
    DEVICE: str = "cuda"

cfg = Config()
print(f"\nConfiguration:")
print(f"  Inference patch: {cfg.INFER_PATCH_SIZE} (same as training)")
print(f"  Batch size: {cfg.BATCH_SIZE} (across {N_GPUS} GPUs)")
print(f"  Threshold: {cfg.THRESHOLD}")
print(f"  TTA: {cfg.TTA_LEVEL}")

# Clear GPU memory on all devices
for i in range(N_GPUS):
    with torch.cuda.device(i):
        torch.cuda.empty_cache()
gc.collect()

In [ ]:
# =============================================================================
# MODEL ARCHITECTURE (MUST MATCH TRAINING EXACTLY)
# =============================================================================

def get_num_groups(channels, max_groups=32):
    for g in [max_groups, 16, 8, 4, 2, 1]:
        if channels % g == 0:
            return g
    return 1


class HybridConv3d(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        mid_ch = out_ch // 2
        self.conv_xy = nn.Conv3d(in_ch, mid_ch, kernel_size=(1, 3, 3), padding=(0, 1, 1), bias=False)
        self.conv_z = nn.Conv3d(in_ch, out_ch - mid_ch, kernel_size=(3, 1, 1), padding=(1, 0, 0), bias=False)
        self.norm = nn.GroupNorm(get_num_groups(out_ch), out_ch)
        self.act = nn.LeakyReLU(0.01, inplace=True)
    def forward(self, x):
        return self.act(self.norm(torch.cat([self.conv_xy(x), self.conv_z(x)], dim=1)))


class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch, use_hybrid=False):
        super().__init__()
        if use_hybrid:
            self.conv = HybridConv3d(in_ch, out_ch)
        else:
            self.conv = nn.Sequential(
                nn.Conv3d(in_ch, out_ch, 3, padding=1, bias=False),
                nn.GroupNorm(get_num_groups(out_ch), out_ch),
                nn.LeakyReLU(0.01, inplace=True),
            )
    def forward(self, x): return self.conv(x)


class MultiScaleResBlock(nn.Module):
    def __init__(self, channels, scales=2, use_hybrid=False):
        super().__init__()
        self.scales = scales
        self.width = channels // scales
        self.convs = nn.ModuleList([ConvBlock(self.width, self.width, use_hybrid=use_hybrid) for _ in range(scales - 1)])
        self.norm = nn.GroupNorm(get_num_groups(channels), channels)
    def forward(self, x):
        splits = torch.chunk(x, self.scales, dim=1)
        outputs = [splits[0]]
        for i, conv in enumerate(self.convs):
            out = conv(splits[i+1] + outputs[-1]) if i > 0 else conv(splits[i+1])
            outputs.append(out)
        return x + self.norm(torch.cat(outputs, dim=1))


class AttentionBlock(nn.Module):
    def __init__(self, channels, reduction=4):
        super().__init__()
        self.gap = nn.AdaptiveAvgPool3d(1)
        self.fc1 = nn.Linear(channels, max(channels // reduction, 8))
        self.fc2 = nn.Linear(max(channels // reduction, 8), channels)
        self.conv_spatial = nn.Conv3d(2, 1, kernel_size=7, padding=3, bias=False)
    def forward(self, x):
        b, c = x.shape[:2]
        ca = torch.sigmoid(self.fc2(F.relu(self.fc1(self.gap(x).view(b, c))))).view(b, c, 1, 1, 1)
        x_ca = x * ca
        sa = torch.sigmoid(self.conv_spatial(torch.cat([x_ca.mean(1, keepdim=True), x_ca.max(1, keepdim=True)[0]], 1)))
        return x_ca * sa


class SurfaceRefinementBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.edge_conv = nn.Conv3d(in_ch, in_ch, 3, padding=1, bias=False)
        self.refine = nn.Sequential(
            nn.Conv3d(in_ch * 2, out_ch, 3, padding=1, bias=False),
            nn.GroupNorm(get_num_groups(out_ch), out_ch),
            nn.LeakyReLU(0.01, inplace=True),
            nn.Conv3d(out_ch, out_ch, 3, padding=1, bias=False),
            nn.GroupNorm(get_num_groups(out_ch), out_ch),
            nn.LeakyReLU(0.01, inplace=True),
        )
    def forward(self, x):
        return self.refine(torch.cat([x, torch.abs(self.edge_conv(x))], dim=1))


class TopoPreservingUNet3D(nn.Module):
    def __init__(self, in_ch=1, out_ch=1, features=None, n_blocks=None,
                 use_attention=True, use_hybrid_conv=True, use_surface_refinement=True,
                 use_deep_supervision=True):  # MUST MATCH TRAINING!
        super().__init__()
        features = features or [32, 64, 128, 256, 320, 320]
        n_blocks = n_blocks or [1, 2, 3, 4, 6, 6]
        self.features = features
        self.use_deep_supervision = use_deep_supervision
        
        self.encoders = nn.ModuleList()
        self.attentions = nn.ModuleList()
        self.pools = nn.ModuleList()
        
        for i, (feat, nb) in enumerate(zip(features, n_blocks)):
            in_channels = in_ch if i == 0 else features[i-1]
            layers = [ConvBlock(in_channels, feat, use_hybrid=use_hybrid_conv and i > 0)]
            layers.extend([MultiScaleResBlock(feat, scales=2, use_hybrid=use_hybrid_conv) for _ in range(nb)])
            self.encoders.append(nn.Sequential(*layers))
            self.attentions.append(AttentionBlock(feat) if use_attention and i >= 2 else nn.Identity())
            if i < len(features) - 1:
                self.pools.append(nn.Conv3d(feat, feat, 2, stride=2))
        
        self.ups = nn.ModuleList()
        self.dec_convs = nn.ModuleList()
        
        for i in range(len(features)-2, -1, -1):
            self.ups.append(nn.ConvTranspose3d(features[i+1], features[i], 2, stride=2))
            if use_surface_refinement and i == 0:
                self.dec_convs.append(SurfaceRefinementBlock(features[i]*2, features[i]))
            else:
                self.dec_convs.append(nn.Sequential(
                    ConvBlock(features[i]*2, features[i], use_hybrid=use_hybrid_conv),
                    MultiScaleResBlock(features[i], scales=2, use_hybrid=use_hybrid_conv),
                ))
        
        self.final = nn.Conv3d(features[0], out_ch, 1)
        
        # DEEP SUPERVISION HEADS - must exist to load checkpoint weights!
        if use_deep_supervision:
            self.ds_heads = nn.ModuleList([
                nn.Conv3d(features[-(i+2)], out_ch, 1) for i in range(min(3, len(features)-1))
            ])
    
    def forward(self, x):
        enc_features = []
        for i, (enc, att) in enumerate(zip(self.encoders, self.attentions)):
            x = att(enc(x))
            enc_features.append(x)
            if i < len(self.pools): x = self.pools[i](x)
        
        enc_features = enc_features[::-1]
        x = enc_features[0]
        
        for i, (up, dec) in enumerate(zip(self.ups, self.dec_convs)):
            x = up(x)
            skip = enc_features[i+1]
            if x.shape[2:] != skip.shape[2:]:
                x = F.interpolate(x, size=skip.shape[2:], mode='trilinear', align_corners=False)
            x = dec(torch.cat([x, skip], dim=1))
        
        return self.final(x)

print("Model loaded (MATCHING TRAINING EXACTLY)")
print("  - use_deep_supervision=True (ds_heads included for checkpoint loading)")

In [ ]:
# =============================================================================
# MULTI-GPU INFERENCE - MATCHING V11 TRAINING EXACTLY
# =============================================================================

def robust_zscore_normalize(img, lower_percentile=0.5, upper_percentile=99.5):
    """
    MUST MATCH TRAINING EXACTLY!
    From V11 training notebook - percentile clipping + Z-score.
    """
    p_low = np.percentile(img, lower_percentile)
    p_high = np.percentile(img, upper_percentile)
    img_clipped = np.clip(img, p_low, p_high)
    mean = img_clipped.mean()
    std = img_clipped.std()
    img_norm = (img_clipped - mean) / (std + 1e-8)
    return img_norm.astype(np.float32)


def create_gaussian_weight(patch_size, sigma=0.125):
    d, h, w = patch_size
    gz = np.exp(-0.5 * ((np.arange(d) - d/2) / (d * sigma)) ** 2)
    gy = np.exp(-0.5 * ((np.arange(h) - h/2) / (h * sigma)) ** 2)
    gx = np.exp(-0.5 * ((np.arange(w) - w/2) / (w * sigma)) ** 2)
    return (gz[:, None, None] * gy[None, :, None] * gx[None, None, :]).astype(np.float32)


def get_patch_positions(volume_shape, patch_size, overlap=0.5):
    """Generate all patch positions for the volume."""
    D, H, W = volume_shape
    pd, ph, pw = patch_size
    sd, sh, sw = int(pd*(1-overlap)), int(ph*(1-overlap)), int(pw*(1-overlap))
    
    z_pos = list(range(0, max(1, D-pd+1), sd))
    if len(z_pos) == 0 or z_pos[-1] + pd < D: z_pos.append(max(0, D - pd))
    y_pos = list(range(0, max(1, H-ph+1), sh))
    if len(y_pos) == 0 or y_pos[-1] + ph < H: y_pos.append(max(0, H - ph))
    x_pos = list(range(0, max(1, W-pw+1), sw))
    if len(x_pos) == 0 or x_pos[-1] + pw < W: x_pos.append(max(0, W - pw))
    
    positions = []
    for z in z_pos:
        for y in y_pos:
            for x in x_pos:
                positions.append((z, y, x))
    return positions


@torch.no_grad()
def sliding_window_inference_multigpu(model, volume, patch_size, overlap=0.5, batch_size=2):
    """
    Multi-GPU sliding window inference.
    Uses 192³ patches to match training.
    """
    model.eval()
    
    D, H, W = volume.shape
    pd, ph, pw = patch_size
    
    # Pad if needed
    pad_d, pad_h, pad_w = max(0, pd-D), max(0, ph-H), max(0, pw-W)
    if pad_d > 0 or pad_h > 0 or pad_w > 0:
        volume = np.pad(volume, ((0,pad_d),(0,pad_h),(0,pad_w)), mode='reflect')
        D, H, W = volume.shape
    
    pred_sum = np.zeros((D, H, W), dtype=np.float32)
    weight_sum = np.zeros((D, H, W), dtype=np.float32)
    gauss = create_gaussian_weight(patch_size)
    
    # Get all patch positions
    positions = get_patch_positions((D, H, W), patch_size, overlap)
    
    # Normalize volume EXACTLY as in training
    vol_norm = robust_zscore_normalize(volume, lower_percentile=0.5, upper_percentile=99.5)
    
    print(f"  Total patches: {len(positions)}, Batch size: {batch_size}")
    
    # Process in batches
    for batch_start in tqdm(range(0, len(positions), batch_size), desc="Inference"):
        batch_end = min(batch_start + batch_size, len(positions))
        batch_positions = positions[batch_start:batch_end]
        
        # Extract patches for this batch
        patches = []
        for (z, y, x) in batch_positions:
            patch = vol_norm[z:z+pd, y:y+ph, x:x+pw]
            patches.append(patch)
        
        # Stack into batch tensor [B, 1, D, H, W]
        batch_tensor = torch.from_numpy(np.stack(patches)[:, None]).cuda().half()
        
        # Forward pass
        with torch.cuda.amp.autocast(dtype=torch.float16):
            batch_pred = torch.sigmoid(model(batch_tensor))
        
        # Convert back to numpy
        batch_pred = batch_pred.squeeze(1).float().cpu().numpy()
        
        # Accumulate predictions
        for i, (z, y, x) in enumerate(batch_positions):
            pred_sum[z:z+pd, y:y+ph, x:x+pw] += batch_pred[i] * gauss
            weight_sum[z:z+pd, y:y+ph, x:x+pw] += gauss
        
        # Cleanup
        del batch_tensor, batch_pred, patches
        
        # Periodic GPU cleanup
        if (batch_start // batch_size) % 10 == 0:
            torch.cuda.empty_cache()
    
    pred = pred_sum / np.maximum(weight_sum, 1e-8)
    
    # Remove padding
    if pad_d > 0: pred = pred[:-pad_d]
    if pad_h > 0: pred = pred[:, :-pad_h]
    if pad_w > 0: pred = pred[:, :, :-pad_w]
    
    return pred


@torch.no_grad()
def inference_with_tta_multigpu(model, volume, patch_size, overlap=0.5, batch_size=2, tta='flip'):
    """TTA with multi-GPU support."""
    # Original
    print("  TTA: Original")
    pred = sliding_window_inference_multigpu(model, volume, patch_size, overlap, batch_size)
    
    if tta == 'none':
        return pred
    
    preds = [pred]
    del pred
    gc.collect()
    for i in range(N_GPUS):
        with torch.cuda.device(i):
            torch.cuda.empty_cache()
    
    if tta in ['flip', 'full']:
        for axis in [0, 1, 2]:
            print(f"  TTA: Flip axis {axis}")
            vol_flip = np.flip(volume, axis).copy()
            pred_flip = sliding_window_inference_multigpu(model, vol_flip, patch_size, overlap, batch_size)
            preds.append(np.flip(pred_flip, axis).copy())
            
            del vol_flip, pred_flip
            gc.collect()
            for i in range(N_GPUS):
                with torch.cuda.device(i):
                    torch.cuda.empty_cache()
    
    print(f"  TTA: Averaging {len(preds)} predictions")
    result = np.mean(preds, axis=0)
    del preds
    gc.collect()
    
    return result

print("Inference functions ready (MATCHING V11 TRAINING)")
print(f"  Normalization: Percentile clipping (0.5-99.5%) + Z-score")
print(f"  Patch size: {cfg.INFER_PATCH_SIZE}")

In [ ]:
# =============================================================================
# POST-PROCESSING: SIMPLE THRESHOLD + TINY COMPONENT REMOVAL
# =============================================================================
# Score history:
#   Skeleton-guided (threshold 0.70) → 0.471 (TOO AGGRESSIVE for V11)
#   Simple threshold 0.50, no post-proc → 0.538
#   Frangi + adaptive threshold → 0.538
#
# V11 model has low mean prob (0.109) - aggressive post-processing hurts.
# Strategy: threshold 0.50 + only remove tiny noise (<50 voxels)

try:
    import cc3d
    USE_CC3D = True
    print("cc3d available - using fast connected components")
except ImportError:
    USE_CC3D = False
    print("cc3d not found, using scipy for connected components")


def connected_components_3d(volume, connectivity=26):
    """Compute 3D connected components."""
    if USE_CC3D:
        return cc3d.connected_components(volume, connectivity=connectivity)
    else:
        from scipy import ndimage
        if connectivity == 26:
            struct = ndimage.generate_binary_structure(3, 3)
        elif connectivity == 6:
            struct = ndimage.generate_binary_structure(3, 1)
        else:
            struct = ndimage.generate_binary_structure(3, 2)
        labeled, _ = ndimage.label(volume, structure=struct)
        return labeled


def simple_postprocess(pred_prob, threshold=0.50, min_component_size=50, verbose=True):
    """
    Minimal post-processing: threshold + remove tiny components only.
    
    V11 model has low confidence (mean ~0.11), so aggressive post-processing
    hurts the score. Keep it simple.
    """
    if verbose:
        print("Post-processing (SIMPLE):")
        print(f"  Input: min={pred_prob.min():.3f}, max={pred_prob.max():.3f}, mean={pred_prob.mean():.3f}")
    
    # Step 1: Threshold
    binary = (pred_prob > threshold).astype(np.uint8)
    if verbose:
        print(f"  1. Threshold ({threshold}): {binary.sum():,} voxels, FG={100*binary.mean():.2f}%")
    
    if binary.sum() == 0:
        if verbose:
            print("  WARNING: Empty mask!")
        return binary
    
    # Step 2: Remove only tiny noise components (< 50 voxels)
    labeled = connected_components_3d(binary)
    n_before = labeled.max()
    
    for i in range(1, n_before + 1):
        component_mask = labeled == i
        if component_mask.sum() < min_component_size:
            binary[component_mask] = 0
    
    labeled_after = connected_components_3d(binary)
    n_after = labeled_after.max()
    
    if verbose:
        print(f"  2. Components: {n_before} -> {n_after} (removed < {min_component_size} voxels)")
        print(f"  Final: {binary.sum():,} voxels, FG={100*binary.mean():.2f}%")
    
    return binary


print("Post-processing ready: SIMPLE (threshold 0.50 + tiny component removal)")
print("  Threshold: 0.50")
print("  Min component size: 50")

In [ ]:
# =============================================================================
# SUBMISSION FORMAT: TIFF MASKS
# =============================================================================

import zipfile

def save_mask_tiff(mask, output_path):
    """Save mask as TIFF - NO compression to match expected file size."""
    mask_uint8 = mask.astype(np.uint8)
    # No compression - raw TIFF
    tifffile.imwrite(str(output_path), mask_uint8, compression=None)
    print(f"  Saved: {output_path} ({mask_uint8.shape}, dtype={mask_uint8.dtype})")

def create_submission_zip(mask_dir, output_zip):
    """Create submission.zip containing all mask TIFFs."""
    with zipfile.ZipFile(output_zip, 'w', zipfile.ZIP_DEFLATED) as zf:
        for tif_path in sorted(Path(mask_dir).glob("*.tif")):
            zf.write(tif_path, tif_path.name)
            print(f"  Added to zip: {tif_path.name}")
    print(f"Submission zip: {output_zip}")

print("TIFF submission functions loaded (no compression)")

In [ ]:
# =============================================================================
# LOAD MODEL WITH MULTI-GPU (DataParallel)
# =============================================================================

# Clear all GPU memory
for i in range(N_GPUS):
    with torch.cuda.device(i):
        torch.cuda.empty_cache()
gc.collect()

print(f"Loading: {cfg.CHECKPOINT_PATH}")

# Create model with SAME config as training (including deep supervision)
model = TopoPreservingUNet3D(
    features=cfg.FEATURES, 
    n_blocks=cfg.N_BLOCKS,
    use_attention=True,
    use_hybrid_conv=True,
    use_surface_refinement=True,
    use_deep_supervision=True,  # MUST be True to load all weights
).cuda()

# Load checkpoint
ckpt = torch.load(cfg.CHECKPOINT_PATH, map_location='cuda:0', weights_only=False)
state = {k.replace('_orig_mod.', ''): v for k, v in ckpt['model_state_dict'].items()}

# Load with strict=True to verify ALL weights match
missing, unexpected = model.load_state_dict(state, strict=False)
if missing:
    print(f"  WARNING - Missing keys ({len(missing)}):")
    for k in missing[:10]:
        print(f"    {k}")
if unexpected:
    print(f"  WARNING - Unexpected keys ({len(unexpected)}):")
    for k in unexpected[:10]:
        print(f"    {k}")
if not missing and not unexpected:
    print(f"  ALL weights loaded perfectly!")

print(f"  Epoch: {ckpt.get('epoch', '?')}, Best Dice: {ckpt.get('best_dice', '?'):.4f}")

# Wrap with DataParallel for multi-GPU
if N_GPUS > 1 and cfg.USE_MULTI_GPU:
    print(f"\n>>> Enabling DataParallel across {N_GPUS} GPUs")
    model = nn.DataParallel(model, device_ids=list(range(N_GPUS)))
    print(f"  Device IDs: {list(range(N_GPUS))}")
else:
    print("\n>>> Single GPU mode")

# Convert to half precision
model.half()
model.eval()

# Memory report
print("\nGPU Memory Usage:")
for i in range(N_GPUS):
    mem = torch.cuda.memory_allocated(i) / 1e9
    print(f"  GPU {i}: {mem:.2f} GB")

print("\nModel ready for inference!")

In [ ]:
# =============================================================================
# RUN INFERENCE (MULTI-GPU) - SIMPLE POST-PROCESSING
# =============================================================================

test_files = sorted(cfg.TEST_ROOT.glob("*.tif")) + sorted(cfg.TEST_ROOT.glob("*.tiff"))
print(f"Found {len(test_files)} test volumes")

# Create output directory for masks
mask_dir = cfg.OUTPUT_DIR / "masks"
mask_dir.mkdir(exist_ok=True)

for vol_path in test_files:
    vol_id = vol_path.stem
    print(f"\n{'='*70}")
    print(f"Processing: {vol_id}")
    print(f"{'='*70}")
    
    # Clear memory on all GPUs before each volume
    for i in range(N_GPUS):
        with torch.cuda.device(i):
            torch.cuda.empty_cache()
    gc.collect()
    
    # Load volume
    volume = tifffile.imread(str(vol_path)).astype(np.float32)
    original_shape = volume.shape
    print(f"Shape: {original_shape}")
    
    # Inference with multi-GPU function
    print(f"Running inference (patch={cfg.INFER_PATCH_SIZE}, TTA={cfg.TTA_LEVEL}, batch={cfg.BATCH_SIZE})...")
    pred_prob = inference_with_tta_multigpu(
        model, volume, cfg.INFER_PATCH_SIZE, cfg.OVERLAP,
        cfg.BATCH_SIZE, cfg.TTA_LEVEL
    )
    
    del volume
    gc.collect()
    
    # Simple post-processing: threshold 0.50 + remove tiny components
    pred_mask = simple_postprocess(
        pred_prob,
        threshold=cfg.THRESHOLD,        # 0.50
        min_component_size=50,           # Only remove tiny noise
        verbose=True,
    )
    
    # Verify shape matches original
    assert pred_mask.shape == original_shape, f"Shape mismatch: {pred_mask.shape} vs {original_shape}"
    
    # Save as TIFF (no compression)
    mask_path = mask_dir / f"{vol_id}.tif"
    save_mask_tiff(pred_mask, mask_path)
    
    # Check file size
    actual_size = mask_path.stat().st_size / 1e6
    print(f"  Saved: {mask_path.name} ({actual_size:.2f} MB)")
    
    # Cleanup
    del pred_prob, pred_mask
    gc.collect()
    for i in range(N_GPUS):
        with torch.cuda.device(i):
            torch.cuda.empty_cache()

print(f"\n{'='*70}")
print("INFERENCE COMPLETE")
print(f"{'='*70}")

In [ ]:
# =============================================================================
# CREATE SUBMISSION ZIP
# =============================================================================

submission_zip = cfg.OUTPUT_DIR / "submission.zip"
create_submission_zip(mask_dir, submission_zip)

# Verify submission
print(f"\nSubmission contents:")
with zipfile.ZipFile(submission_zip, 'r') as zf:
    for info in zf.infolist():
        print(f"  {info.filename}: {info.file_size / 1e6:.2f} MB")

print(f"\n>>> Ready to submit: {submission_zip}")